# VLM From Scratch

### Part 1: Setup

In [ ]:
# Package Imports
!uv pip install -q transformers datasets torch torchvision pillow accelerate einops timm

In [ ]:
# Configs 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    ViTModel,
    ViTImageProcessor,
)
from datasets import load_dataset
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Print torch version
print(f"Torch version :{torch.__version__}")

In [ ]:
# Image format
%config InlineBackend.figure_format = 'retina'

### Vision Encoder (Vit)

the basic job of vit is to convert the images into a sequence of meaningful vectors.

Example: For a 224x224 image with 16x16 patches, we get: - (224/16) x (224/16) = 14 x 14 = 196 patches - Plus 1 CLS token = 197 tokens total - Each token has dimension 1024 (for ViT-Large)

In [ ]:
# Load pretrained ViT-Large (larger than ViT-Base for better performance)
vision_model_name = "google/vit-large-patch16-224"
vision_encoder = ViTModel.from_pretrained(vision_model_name)
image_processor = ViTImageProcessor.from_pretrained(vision_model_name)

# Freeze the vision encoder - we won't train it
for param in vision_encoder.parameters():
    param.requires_grad = False

vision_encoder = vision_encoder.to(device)
vision_encoder.eval()

print(f"Vision encoder hidden size: {vision_encoder.config.hidden_size}")
print(f"Number of patches: {(224//16)**2} + 1 CLS token = {(224//16)**2 + 1} tokens")

In [ ]:
# Test the vision encoder with a sample image
url = "https://image.api.playstation.com/vulcan/ap/rnd/202603/2506/93f66b17d66159f2a06f2f001b0e28cb485b524c9204797b.png"
response = requests.get(url)
sample_image = Image.open(BytesIO(response.content))

plt.figure(figsize=(6, 6))
plt.imshow(sample_image)
plt.title("Sample Image from COCO")
plt.axis('off')
plt.show()

# Process the image
inputs = image_processor(sample_image, return_tensors="pt").to(device)

with torch.no_grad():
    vision_outputs = vision_encoder(**inputs)

# Get the sequence of patch features (excluding pooler output)
image_features = vision_outputs.last_hidden_state
print(f"Image features shape: {image_features.shape}")
print(f"  - Batch size: {image_features.shape[0]}")
print(f"  - Sequence length (patches + CLS): {image_features.shape[1]}")
print(f"  - Hidden dimension: {image_features.shape[2]}")

### Part 2: Language Model

Model Chosen: SmolLM-360M ( Small but an effective model for testing/ experimenting with basics)

In [ ]:
# Load the language model (SmolLM-360M for better performance)
lm_model_name = "HuggingFaceTB/SmolLM-360M"
tokenizer = AutoTokenizer.from_pretrained(lm_model_name)
language_model = AutoModelForCausalLM.from_pretrained(lm_model_name)

# Add padding token if not present
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

language_model = language_model.to(device)

print(f"Language model hidden size: {language_model.config.hidden_size}")
print(f"Vocabulary size: {language_model.config.vocab_size}")
print(f"Number of layers: {language_model.config.num_hidden_layers}")

In [ ]:
# Test the language model
test_text = "A photo of"
inputs = tokenizer(test_text, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = language_model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.pad_token_id
    )

print(f"Generated text: {tokenizer.decode(outputs[0], skip_special_tokens=True)}")

### Part 3: Projection Layer

- Acts as the bridge between the vision and langauge models.
- From: Vision features (batch, 197, 1024) - To: Language-compatible embeddings (batch, 197, 960)
- Lets use a simple multi-layer projection with: 1. Linear layer (1024 -> 960) 2. GELU activation 3. LayerNorm for stability

In [ ]:
class VisionProjector(nn.Module):
    """Projects vision features into the language model's embedding space."""
    
    def __init__(self, vision_dim: int, language_dim: int):
        super().__init__()
        self.projection = nn.Sequential(
            nn.Linear(vision_dim, language_dim),
            nn.GELU(),
            nn.LayerNorm(language_dim),
            nn.Linear(language_dim, language_dim),  # Additional layer for better alignment
        )
    
    def forward(self, vision_features: torch.Tensor) -> torch.Tensor:
        """Project vision features to language embedding space.
        
        Args:
            vision_features: (batch, seq_len, vision_dim)
        Returns:
            projected: (batch, seq_len, language_dim)
        """
        return self.projection(vision_features)


# Create projector
vision_dim = vision_encoder.config.hidden_size  # 1024 (ViT-Large)
language_dim = language_model.config.hidden_size  # 960 (SmolLM-360M)

projector = VisionProjector(vision_dim, language_dim).to(device)

print(f"Projector: {vision_dim} -> {language_dim}")
print(f"Trainable parameters: {sum(p.numel() for p in projector.parameters()):,}")

In [ ]:
# Test the projector
with torch.no_grad():
    projected_features = projector(image_features)

print(f"Original vision features: {image_features.shape}")
print(f"Projected features: {projected_features.shape}")

### Part 4: The whole Architecture

Flow:
    Vision Encoder -> Projection -> Embed Texts -> Concact (img_embed, text_embed) -> Generate 

In [ ]:
class MiniVLM(nn.Module):

  def __init__(self, vision_encoder: ViTModel, language_model: AutoModelForCausalLM, projector: VisionProjector, tokenizer: AutoTokenizer):
    super().__init__()
    self.vision_encoder = vision_encoder
    self.language_model = language_model
    self.projector = projector
    self.tokenizer = tokenizer
    # Freeze vision encoder
    for param in self.vision_encoder.parameters():
        param.requires_grad = False
  
  def encode_image(self, image: torch.Tensor) -> torch.Tensor:
    # Encode the images to the standard vector space
    with torch.no_grad():
      vision_outputs = self.vision_encoder(image)
    projected = vision_outputs.last_hidden_state
    return self.projector(projected)
  
  def forward(self,
        pixel_values: torch.Tensor,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        labels: torch.Tensor = None,):
    """Forward pass for training.
        
        Args:
            pixel_values: Image tensor (batch, 3, 224, 224)
            input_ids: Text token IDs (batch, text_len)
            attention_mask: Attention mask for text (batch, text_len)
            labels: Target token IDs for loss computation (batch, text_len)
        """
    batch_size = pixel_values.shape[0]
    # 1. Encode and embedd image
    image_embeds = self.encode_image(pixel_values)
    num_image_tokens = image_embeds.shape[1]

    # 2. Text embeds from LLM
    text_embeds = self.language_model.get_input_embeddings()(input_ids)

    # 3. Concat image and text embeds
    combined_embeds = torch.cat([image_embeds, text_embeds], dim=1)

    # 4. Create attention mask for combined sequence
    img_attention = torch.ones((batch_size, num_image_tokens), dtype=attention_mask.dtype, device=attention_mask.device)
    combined_attention = torch.cat([img_attention, attention_mask], dim=1)

    # 5. Create labels: -100 for image tokens (ignore in loss), actual labels for text
    if labels is not None:
        image_labels = torch.full(
            (batch_size, num_image_tokens),
            fill_value=-100,  # -100 is ignored by CrossEntropyLoss
            dtype=labels.dtype,
            device=labels.device
        )
        combined_labels = torch.cat([image_labels, labels], dim=1)
    else:
        combined_labels = None
    
    # 6. Forward through language model
    outputs = self.language_model(
        inputs_embeds=combined_embeds,
        attention_mask=combined_attention,
        labels=combined_labels,
        return_dict=True,
    )
    
    return outputs
  
  @torch.no_grad()
  def generate(self, pixel_values: torch.Tensor, max_new_tokens: int = 50, temperatue: float = 0.7, do_sample: bool = True) -> str:
    """Generate a caption for an image."""
    self.eval()
    
    # Encode image
    image_embeds = self.encode_image(pixel_values)  # (1, 197, hidden)
    
    # Start with a prompt
    prompt = "This image shows"
    prompt_ids = self.tokenizer.encode(prompt, return_tensors="pt").to(pixel_values.device)
    prompt_embeds = self.language_model.get_input_embeddings()(prompt_ids)
    
    # Combine image and prompt embeddings
    combined_embeds = torch.cat([image_embeds, prompt_embeds], dim=1)
    
    # Generate token by token
    generated_ids = prompt_ids.clone()

    for _ in range(max_new_tokens):
      # Get current embeddings
      current_embeds = self.language_model.get_input_embeddings()(generated_ids)
      full_embeds = torch.cat([image_embeds, current_embeds], dim=1)

      # Forward
      outputs = self.language_model(inputs_embeds=full_embeds)
      next_token_logits = outputs.logits[:, -1, :]

      # Sample or greedy
      if do_sample:
        next_token_id = torch.multinomial(F.softmax(next_token_logits / temperatue, dim=-1), num_samples=1)
      else:
        next_token_id = torch.argmax(next_token_logits, dim=-1, keepdim=True)
    
      generated_ids = torch.cat([generated_ids, next_token_id], dim=1)

      # Stop if EOS
      if next_token_id.item() == self.tokenizer.eos_token_id:
        break
    
    # Decode and return
    return self.tokenizer.decode(generated_ids[0], skip_special_tokens=True)

# Create the VLM
vlm = MiniVLM(
    vision_encoder=vision_encoder,
    language_model=language_model,
    projector=projector,
    tokenizer=tokenizer,
)
vlm = vlm.to(device)

# Count parameters
total_params = sum(p.numel() for p in vlm.parameters())
trainable_params = sum(p.numel() for p in vlm.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}")

### Part 5: Load the dataset ( flickr8k )

We use a subset of the dataset, its a popular dataset for image captioning benchmark

In [ ]:
# Load Flickr8k captions dataset
dataset = load_dataset("jxie/flickr8k", split="train")

# Use 2000 samples for better training
num_samples = 2000
dataset = dataset.shuffle(seed=42).select(range(num_samples))

print(f"Dataset size: {len(dataset)}")
print(f"Sample item keys: {dataset[0].keys()}")

In [ ]:
# Look at a sample - Flickr8k has caption_0 through caption_4
sample = dataset[0]
print(f"Image: {type(sample['image'])}")
print(f"Captions:")
for i in range(5):
    print(f"  {i}: {sample[f'caption_{i}']}")

plt.figure(figsize=(6, 6))
plt.imshow(sample['image'])
plt.title(f"Caption: {sample['caption_0'][:60]}...")
plt.axis('off')
plt.show()

In [ ]:
import random

class Flickr8kDataset(Dataset):
    """Dataset for Flickr8k image-caption pairs."""
    
    def __init__(self, hf_dataset, image_processor, tokenizer, max_length=64):
        self.dataset = hf_dataset
        self.image_processor = image_processor
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        item = self.dataset[idx]
        
        # Process image
        image = item['image'].convert('RGB')
        pixel_values = self.image_processor(image, return_tensors="pt").pixel_values.squeeze(0)
        
        # Randomly select one of the 5 captions
        caption_idx = random.randint(0, 4)
        caption = item[f'caption_{caption_idx}']
        
        # Tokenize caption
        encoding = self.tokenizer(
            caption,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        input_ids = encoding['input_ids'].squeeze(0)
        attention_mask = encoding['attention_mask'].squeeze(0)
        
        # Labels: mask padding tokens with -100
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100
        
        return {
            'pixel_values': pixel_values,
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
        }


# Create dataset and dataloader
train_dataset = Flickr8kDataset(dataset, image_processor, tokenizer)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,  # Small batch for memory
    shuffle=True,
    num_workers=0,  # Avoid multiprocessing issues
)

print(f"Number of batches: {len(train_loader)}")

### Baseline comparision

As the model is based on random weights, it has no way to understand the image so the expected output is that it generates delusional/ hallucinated data.

In [ ]:
def generate_caption(model, image, image_processor, device):
    """Generate a caption for an image."""
    model.eval()
    
    # Process image
    if isinstance(image, str):
        # URL
        response = requests.get(image)
        image = Image.open(BytesIO(response.content)).convert('RGB')
    elif not isinstance(image, Image.Image):
        image = Image.fromarray(image)
    
    pixel_values = image_processor(image, return_tensors="pt").pixel_values.to(device)
    
    # Generate
    caption = model.generate(
        pixel_values,
        max_new_tokens=30,
        temperature=0.8,
        do_sample=True,
    )
    
    return caption, image

In [ ]:
# Store some test images and their ground truth captions for before/after comparison
test_indices = [0, 10, 50, 100]
test_samples = [(dataset[i]['image'].convert('RGB'), dataset[i]['caption_0']) for i in test_indices]

# Generate captions with UNTRAINED model
print("=" * 60)
print("UNTRAINED MODEL PREDICTIONS (Random projector weights)")
print("=" * 60)

untrained_captions = []
for i, (image, gt_caption) in enumerate(test_samples):
    caption, _ = generate_caption(vlm, image, image_processor, device)
    untrained_captions.append(caption)
    print(f"\nImage {i+1}:")
    print(f"  Generated: {caption}")
    print(f"  GT: {gt_caption[:80]}...")